# 06 - Machine Learning Data Preparation

### Objective

This notebook prepares the feature-engineered stock datasets for machine learning.

The main steps include:

- Creating 1-day, 5-day, and 20-day movement targets
- Including adjusted price information
- Removing rows with insufficient historical information
- Preventing future information leakage
- Generating ML-ready datasets for all stocks and prediction horizons
- Verifying missing values
- Analyzing target class distributions

### Target Definition

For each prediction horizon:

- **1D** → 1 if the closing price after 1 trading day is higher than today's closing price
- **5D** → 1 if the closing price after 5 trading days is higher than today's closing price
- **20D** → 1 if the closing price after 20 trading days is higher than today's closing price

Otherwise:

- **0** → Future closing price is lower than or equal to today's closing price

In [21]:
import os

os.makedirs("../data/ml_ready", exist_ok=True)

In [1]:
import sys

sys.path.append("../src")

from prepare_ml_data import prepare_ml_data
from config import FEATURES

In [23]:
stocks = [
    "AAPL",
    "MSFT",
    "NVDA",
    "AMZN",    
    "RELIANCE.NS",
    "TCS.NS",
    "HDFCBANK.NS",
    "INFY.NS"
]

## Prepare ML-Ready Datasets

The `prepare_ml_data()` function creates horizon-specific movement targets and removes rows that cannot be used for machine learning.

The same pipeline is applied to all 8 US and Indian stocks across three prediction horizons:

- 1-day (`1D`)
- 5-day (`5D`)
- 20-day (`20D`)

In [24]:
horizons = ["1D", "5D", "20D"]

ml_data = {}

for horizon in horizons:

    ml_data[horizon] = {}

    for stock in stocks:

        ml_data[horizon][stock] = prepare_ml_data(
            f"../data/processed/{stock}_features.csv",
            f"../data/ml_ready/{stock}_{horizon}_ml.csv",
            horizon=horizon
        )

        print(
            f"{stock} - {horizon} ML data preparation completed."
        )
    
    print()

AAPL - 1D ML data preparation completed.
MSFT - 1D ML data preparation completed.
NVDA - 1D ML data preparation completed.
AMZN - 1D ML data preparation completed.
RELIANCE.NS - 1D ML data preparation completed.
TCS.NS - 1D ML data preparation completed.
HDFCBANK.NS - 1D ML data preparation completed.
INFY.NS - 1D ML data preparation completed.

AAPL - 5D ML data preparation completed.
MSFT - 5D ML data preparation completed.
NVDA - 5D ML data preparation completed.
AMZN - 5D ML data preparation completed.
RELIANCE.NS - 5D ML data preparation completed.
TCS.NS - 5D ML data preparation completed.
HDFCBANK.NS - 5D ML data preparation completed.
INFY.NS - 5D ML data preparation completed.

AAPL - 20D ML data preparation completed.
MSFT - 20D ML data preparation completed.
NVDA - 20D ML data preparation completed.
AMZN - 20D ML data preparation completed.
RELIANCE.NS - 20D ML data preparation completed.
TCS.NS - 20D ML data preparation completed.
HDFCBANK.NS - 20D ML data preparation compl

In [25]:
df = ml_data["1D"]["AAPL"]

df.head()

,Date,Adj Close,Close,High,Low,Open,Volume,Company,Daily_Return,Adj_Daily_Return,...,EMA_20,Volatility_20,Volume_Change,High_Low_Range,Open_Close_Change,RSI_14,MACD,MACD_Signal,MACD_Hist,Target_1D
0,2015-03-16,27.734209,31.237499,31.237499,30.717501,30.969999,143497200,AAPL,0.011004,0.011004,...,31.352804,0.013712,-0.307811,0.016647,0.008637,32.710716,0.229702,0.511192,-0.281490,1
1,2015-03-17,28.198105,31.760000,31.830000,31.412500,31.475000,204092400,AAPL,0.016727,0.016726,...,31.391584,0.014193,0.422274,0.013145,0.009055,45.533452,0.240975,0.457149,-0.216174,1
2,2015-03-18,28.515507,32.117500,32.290001,31.592501,31.750000,261083600,AAPL,0.011256,0.011256,...,31.460719,0.014339,0.279242,0.021717,0.011575,44.971639,0.275579,0.420835,-0.145255,0
3,2015-03-19,28.300217,31.875000,32.312500,31.850000,32.187500,183238000,AAPL,-0.007550,-0.007550,...,31.500175,0.014433,-0.298164,0.014510,-0.009709,47.391285,0.280206,0.392709,-0.112503,0
4,2015-03-20,27.945072,31.475000,32.099998,31.290001,32.062500,274780400,AAPL,-0.012549,-0.012549,...,31.497777,0.014538,0.499582,0.025735,-0.018324,41.765631,0.248728,0.363913,-0.115184,1


In [26]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nTarget column:")
print("Target_1D")

Shape: (2895, 26)

Columns:
['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'Company', 'Daily_Return', 'Adj_Daily_Return', 'Return_Lag_1', 'Return_Lag_2', 'Return_Lag_3', 'SMA_5', 'SMA_20', 'SMA_50', 'EMA_20', 'Volatility_20', 'Volume_Change', 'High_Low_Range', 'Open_Close_Change', 'RSI_14', 'MACD', 'MACD_Signal', 'MACD_Hist', 'Target_1D']

Target column:
Target_1D


In [27]:
df.columns

Index(['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume',
       'Company', 'Daily_Return', 'Adj_Daily_Return', 'Return_Lag_1',
       'Return_Lag_2', 'Return_Lag_3', 'SMA_5', 'SMA_20', 'SMA_50', 'EMA_20',
       'Volatility_20', 'Volume_Change', 'High_Low_Range', 'Open_Close_Change',
       'RSI_14', 'MACD', 'MACD_Signal', 'MACD_Hist', 'Target_1D'],
      dtype='object')

## Feature Verification

The ML-ready datasets should contain the adjusted price information introduced during feature engineering.

In particular:

- `Adj Close` represents the adjusted closing price.
- `Adj_Daily_Return` represents the daily percentage change in adjusted closing price.
- `Target_1D`, `Target_5D`, and `Target_20D` are created using future closing-price information.
- Target columns must not be included as input features.

In [28]:
required_features = [
    "Adj Close",
    "Adj_Daily_Return"
]

missing_features = [
    feature
    for feature in required_features
    if feature not in df.columns
]

if missing_features:
    print("Missing features:", missing_features)
else:
    print("All new adjusted-price features are present.")

All new adjusted-price features are present.


In [29]:
for horizon in horizons:

    print(f"\n{horizon} Target values:")
    print(ml_data[horizon]["AAPL"][f"Target_{horizon}"].value_counts())

    print(f"\n{horizon} Target data type:")
    print(ml_data[horizon]["AAPL"][f"Target_{horizon}"].dtype)


1D Target values:
Target_1D
1    1530
0    1365
Name: count, dtype: Int64

1D Target data type:
Int64

5D Target values:
Target_5D
1    1655
0    1236
Name: count, dtype: Int64

5D Target data type:
Int64

20D Target values:
Target_20D
1    1801
0    1075
Name: count, dtype: Int64

20D Target data type:
Int64


## Missing Value Verification

Rows with insufficient historical information are removed during ML data preparation.

The final datasets should therefore contain no missing values.

In [30]:
for horizon in horizons:

    print(f"\n{horizon}")

    print(ml_data[horizon]["AAPL"].isnull().sum())


1D
Date                 0
Adj Close            0
Close                0
High                 0
Low                  0
Open                 0
Volume               0
Company              0
Daily_Return         0
Adj_Daily_Return     0
Return_Lag_1         0
Return_Lag_2         0
Return_Lag_3         0
SMA_5                0
SMA_20               0
SMA_50               0
EMA_20               0
Volatility_20        0
Volume_Change        0
High_Low_Range       0
Open_Close_Change    0
RSI_14               0
MACD                 0
MACD_Signal          0
MACD_Hist            0
Target_1D            0
dtype: int64

5D
Date                 0
Adj Close            0
Close                0
High                 0
Low                  0
Open                 0
Volume               0
Company              0
Daily_Return         0
Adj_Daily_Return     0
Return_Lag_1         0
Return_Lag_2         0
Return_Lag_3         0
SMA_5                0
SMA_20               0
SMA_50               0
EMA_20       

In [35]:
for horizon in horizons:

    print(f"\n========== {horizon} ==========")

    for stock in stocks:

        df = ml_data[horizon][stock]

        print(stock, "| Shape:", df.shape, "| Missing:", df.isnull().sum().sum())


========== 1D ==========
AAPL | Shape: (2895, 26) | Missing: 0
MSFT | Shape: (2895, 26) | Missing: 0
NVDA | Shape: (2895, 26) | Missing: 0
AMZN | Shape: (2895, 26) | Missing: 0
RELIANCE.NS | Shape: (2841, 26) | Missing: 0
TCS.NS | Shape: (2841, 26) | Missing: 0
HDFCBANK.NS | Shape: (2841, 26) | Missing: 0
INFY.NS | Shape: (2841, 26) | Missing: 0

========== 5D ==========
AAPL | Shape: (2891, 26) | Missing: 0
MSFT | Shape: (2891, 26) | Missing: 0
NVDA | Shape: (2891, 26) | Missing: 0
AMZN | Shape: (2891, 26) | Missing: 0
RELIANCE.NS | Shape: (2838, 26) | Missing: 0
TCS.NS | Shape: (2838, 26) | Missing: 0
HDFCBANK.NS | Shape: (2838, 26) | Missing: 0
INFY.NS | Shape: (2838, 26) | Missing: 0

========== 20D ==========
AAPL | Shape: (2876, 26) | Missing: 0
MSFT | Shape: (2876, 26) | Missing: 0
NVDA | Shape: (2876, 26) | Missing: 0
AMZN | Shape: (2876, 26) | Missing: 0
RELIANCE.NS | Shape: (2823, 26) | Missing: 0
TCS.NS | Shape: (2823, 26) | Missing: 0
HDFCBANK.NS | Shape: (2823, 26) | Miss

In [32]:
for horizon in horizons:

    print(f"\n========== {horizon} ==========")

    target = f"Target_{horizon}"

    for stock in stocks:

        print(f"\n{stock}")

        print(ml_data[horizon][stock][target].value_counts())


========== 1D ==========

AAPL
Target_1D
1    1530
0    1365
Name: count, dtype: Int64

MSFT
Target_1D
1    1537
0    1358
Name: count, dtype: Int64

NVDA
Target_1D
1    1569
0    1326
Name: count, dtype: Int64

AMZN
Target_1D
1    1539
0    1356
Name: count, dtype: Int64

RELIANCE.NS
Target_1D
1    1468
0    1373
Name: count, dtype: Int64

TCS.NS
Target_1D
0    1422
1    1419
Name: count, dtype: Int64

HDFCBANK.NS
Target_1D
1    1450
0    1391
Name: count, dtype: Int64

INFY.NS
Target_1D
1    1437
0    1404
Name: count, dtype: Int64

========== 5D ==========

AAPL
Target_5D
1    1655
0    1236
Name: count, dtype: Int64

MSFT
Target_5D
1    1685
0    1206
Name: count, dtype: Int64

NVDA
Target_5D
1    1715
0    1176
Name: count, dtype: Int64

AMZN
Target_5D
1    1640
0    1251
Name: count, dtype: Int64

RELIANCE.NS
Target_5D
1    1552
0    1286
Name: count, dtype: Int64

TCS.NS
Target_5D
1    1465
0    1373
Name: count, dtype: Int64

HDFCBANK.NS
Target_5D
1    1533
0    1305
Name: cou

In [33]:
for horizon in horizons:

    print(f"\n========== {horizon} ==========")

    target = f"Target_{horizon}"

    for stock in stocks:

        print(f"\n{stock}")

        print(ml_data[horizon][stock][target].value_counts(normalize=True) * 100)


========== 1D ==========

AAPL
Target_1D
1    52.849741
0    47.150259
Name: proportion, dtype: Float64

MSFT
Target_1D
1    53.091537
0    46.908463
Name: proportion, dtype: Float64

NVDA
Target_1D
1    54.196891
0    45.803109
Name: proportion, dtype: Float64

AMZN
Target_1D
1    53.160622
0    46.839378
Name: proportion, dtype: Float64

RELIANCE.NS
Target_1D
1    51.671946
0    48.328054
Name: proportion, dtype: Float64

TCS.NS
Target_1D
0    50.052798
1    49.947202
Name: proportion, dtype: Float64

HDFCBANK.NS
Target_1D
1    51.038367
0    48.961633
Name: proportion, dtype: Float64

INFY.NS
Target_1D
1    50.580781
0    49.419219
Name: proportion, dtype: Float64

========== 5D ==========

AAPL
Target_5D
1    57.246627
0    42.753373
Name: proportion, dtype: Float64

MSFT
Target_5D
1    58.284331
0    41.715669
Name: proportion, dtype: Float64

NVDA
Target_5D
1    59.322034
0    40.677966
Name: proportion, dtype: Float64

AMZN
Target_5D
1    56.727776
0    43.272224
Name: proporti

In [36]:
for horizon in horizons:

    df = ml_data[horizon]["AAPL"]

    available_features = [
        feature
        for feature in FEATURES
        if feature in df.columns
    ]

    missing_features = [
        feature
        for feature in FEATURES
        if feature not in df.columns
    ]

    print(f"\n========== {horizon} ==========")

    print("Available ML features:")
    print(available_features)

    print("Missing ML features:")
    print(missing_features)


========== 1D ==========
Available ML features:
['Adj Close', 'Adj_Daily_Return', 'Daily_Return', 'Return_Lag_1', 'Return_Lag_2', 'Return_Lag_3', 'SMA_5', 'SMA_20', 'SMA_50', 'EMA_20', 'Volatility_20', 'Volume_Change', 'High_Low_Range', 'Open_Close_Change', 'RSI_14', 'MACD', 'MACD_Signal', 'MACD_Hist']
Missing ML features:
[]

========== 5D ==========
Available ML features:
['Adj Close', 'Adj_Daily_Return', 'Daily_Return', 'Return_Lag_1', 'Return_Lag_2', 'Return_Lag_3', 'SMA_5', 'SMA_20', 'SMA_50', 'EMA_20', 'Volatility_20', 'Volume_Change', 'High_Low_Range', 'Open_Close_Change', 'RSI_14', 'MACD', 'MACD_Signal', 'MACD_Hist']
Missing ML features:
[]

========== 20D ==========
Available ML features:
['Adj Close', 'Adj_Daily_Return', 'Daily_Return', 'Return_Lag_1', 'Return_Lag_2', 'Return_Lag_3', 'SMA_5', 'SMA_20', 'SMA_50', 'EMA_20', 'Volatility_20', 'Volume_Change', 'High_Low_Range', 'Open_Close_Change', 'RSI_14', 'MACD', 'MACD_Signal', 'MACD_Hist']
Missing ML features:
[]


## Conclusion

The feature-engineered datasets were successfully transformed into ML-ready datasets for all 8 stocks across three prediction horizons: 1-day, 5-day, and 20-day.

The preparation process included:

- Creating horizon-specific binary movement targets
- Removing observations with insufficient historical data
- Removing the temporary future closing-price column to prevent data leakage
- Verifying missing values
- Examining target class distributions
- Verifying the availability of all ML features

The resulting datasets are now ready for chronological train-test splitting and time-series cross-validation during the machine learning model development stage.